# Evaluation: with and without retrieval
A fixed set of 18 questions with reference answers, written from the pages: 15 are answered by one page, and 3 are about things the documentation does not cover.

- **Retrieval:** whether the expected page is in the top 5, and at which rank, for both chunk profiles.
- **Answers:** the same model answers with retrieval (top 5, each profile) and without it. A second model judges each answer against the reference:
  - `correct`: matches the reference
  - `relevant`: addresses the question
  - `grounded`: supported by the retrieved context
- **Missing context:** the RAG prompt must answer `I don't know based on the documentation.` when the context lacks the answer. This should happen on the out-of-scope questions and not on the others.

Every call is logged to `qa_log`, and the results go to `eval_results`.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("src_path", "")

import sys

sys.path.insert(0, dbutils.widgets.get("src_path"))
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}"

dbutils.widgets.text("llm_endpoint", "")

from rag_kb.service import Assistant, log_interactions

assistant = Assistant(f"{target}.docs_chunks_index", dbutils.widgets.get("llm_endpoint"))

dbutils.widgets.text("judge_endpoint", "")
dbutils.widgets.text("top_k", "5")

import uuid

from rag_kb.eval_set import QUESTIONS
from rag_kb.qa import build_judge_messages, format_context, hit_rank, parse_judgement
from rag_kb.text import PROFILES

judge_endpoint = dbutils.widgets.get("judge_endpoint")
top_k = int(dbutils.widgets.get("top_k"))
run_id = str(uuid.uuid4())

In [ ]:
results, logs = [], []
for q in QUESTIONS:
    runs = [assistant.ask(q["question"], True, top_k, profile, source="evaluation") for profile in PROFILES]
    runs.append(assistant.ask(q["question"], False, source="evaluation"))
    for r in runs:
        context = format_context(r["chunks"]) if r["chunks"] else None
        verdict, _ = assistant.chat(build_judge_messages(q["question"], q["reference"], r["answer"], context), endpoint=judge_endpoint, max_tokens=60)
        scores = parse_judgement(verdict)
        results.append({
            "run_id": run_id, "question_id": q["id"], "in_scope": q["doc"] is not None, "mode": r["mode"],
            "chunk_profile": r["chunk_profile"], "expected_doc": q["doc"],
            "hit_rank": hit_rank(q["doc"], r["retrieved_docs"]) if q["doc"] and r["mode"] == "rag" else None,
            "abstained": r["abstained"], **scores, "latency_ms": r["latency_ms"],
            "prompt_tokens": r["prompt_tokens"], "completion_tokens": r["completion_tokens"], "answer": r["answer"],
        })
        logs.append(r)

log_interactions(spark, f"{target}.qa_log", logs)
(
    spark.createDataFrame(results, "run_id string, question_id string, in_scope boolean, mode string, chunk_profile string, expected_doc string, hit_rank int, abstained boolean, correct int, relevant int, grounded int, latency_ms int, prompt_tokens int, completion_tokens int, answer string")
    .selectExpr("*", "current_timestamp() AS evaluated_at")
    .write.mode("append").saveAsTable(f"{target}.eval_results")
)

In [ ]:
summary = spark.sql(f"""
    SELECT mode, coalesce(chunk_profile, '-') AS chunk_profile,
           round(avg(CASE WHEN in_scope THEN CAST(hit_rank = 1 AS INT) END), 2) AS hit_at_1,
           round(avg(CASE WHEN in_scope THEN CAST(hit_rank IS NOT NULL AS INT) END), 2) AS hit_at_k,
           round(avg(CASE WHEN in_scope THEN correct END), 2) AS correct_in_scope,
           round(avg(CASE WHEN in_scope THEN relevant END), 2) AS relevant_in_scope,
           round(avg(CASE WHEN in_scope THEN grounded END), 2) AS grounded_in_scope,
           round(avg(CASE WHEN in_scope THEN CAST(abstained AS INT) END), 2) AS abstained_in_scope,
           round(avg(CASE WHEN NOT in_scope THEN correct END), 2) AS declined_out_of_scope,
           round(avg(latency_ms)) AS avg_latency_ms,
           round(avg(prompt_tokens + completion_tokens)) AS avg_tokens
    FROM {target}.eval_results WHERE run_id = '{run_id}'
    GROUP BY ALL ORDER BY mode DESC, chunk_profile DESC
""")
display(summary)

In [ ]:
display(spark.sql(f"""
    SELECT question_id, mode, chunk_profile, hit_rank, correct, grounded, abstained, left(answer, 200) AS answer
    FROM {target}.eval_results
    WHERE run_id = '{run_id}' AND (correct = 0 OR (in_scope AND hit_rank IS NULL AND mode = 'rag'))
    ORDER BY question_id, mode
"""))